In [ ]:
import numpy as np
import readgadget
import bigfile
import MAS_library as MASL
import Pk_library as PKL
import matplotlib.pyplot as plt
from pylab import *
import dddf

# settings
realization = 0
N_p = 128
grid_size = N_p
if (N_p == 128) | (N_p == 256):
    snapshots = [
        f"../FastPM/L1N{N_p}fnl0r1000{realization + 1}/dm/dm_0.0100/1",
        f"../FastPM/L1N{N_p}fnl0r1000{realization + 1}/dm/dm_1.0000/1",
    ]
    init_snapshot_z = 99.0
elif N_p == 512:
    snapshots = [
        f"../Quijote/fiducial/{realization}/ICs/ics",
        f"../Quijote/fiducial/{realization}/snapdir_004/snap_004",
    ]
    init_snapshot_z = 127.0
final_snapshot_z = 0.0
Omega_m = 0.3175
MAS = "CIC"
threads = 16
# padding = 3
fit_layer = 2
# boxsize = np.astype(readgadget.header(
#     snapshots[0]).boxsize / 1e3, real_dtype)  # Mpc/h
boxsize = 1000.0  # Mpc/h
grid_width = boxsize / grid_size
dl = dddf.DDDF(init_snapshot_z, final_snapshot_z, Omega_m, threads)
veck_main = dl.Veck(dl, N_p, boxsize, padding=0)
rng = np.random.default_rng(seed=42)

snapshot_info = [
    dl.get_snapshot(
        snapshot,
        "bigfile" if (N_p == 128) | (N_p == 256) else "gadget",
        boxsize,
        grid_size,
    )
    for snapshot in snapshots
]

target_par_disp = np.astype(
    snapshot_info[1]["pos"] - snapshot_info[0]["pos"], dl.real_dtype
)
target_par_disp = np.where(
    target_par_disp < boxsize / 2, target_par_disp, target_par_disp - boxsize
)
target_par_disp = np.where(
    target_par_disp > -boxsize / 2, target_par_disp, target_par_disp + boxsize
)

init_delta = snapshot_info[0]["delta"]


target_disp_field = np.zeros((N_p, N_p, N_p, 3), dtype=dl.real_dtype)
dl.disp_from_par(
    target_disp_field, snapshot_info[0]["pos"], target_par_disp, N_p, boxsize
)

# par_disp = np.zeros((N_p**3, 3), dtype=dl.real_dtype)
# dl.assign_disp(par_disp, snapshot_info[0]["pos"], target_disp_field, N_p, boxsize)
# final_par_pos = snapshot_info[0]["pos"] + par_disp
# final_par_pos %= boxsize

target_psi_div = dl.divergence(target_disp_field, veck_main)

psi_div_1 = dl.div_psi_1(init_delta)
psi_div_2 = dl.div_psi_2(init_delta, veck_main)
ZA_disp = dl.disp_from_psi_div(psi_div_1, veck_main, N_p)
LPT2_disp = dl.disp_from_psi_div(psi_div_1 + psi_div_2, veck_main, N_p)

In [ ]:
def drawfigs():
    plt.cla()

    labels = ["N-body", "ZA"]
    final_pos = [
        snapshot_info[1]["pos"],
        dl.par_pos_from_psi_div(
            psi_div_1, snapshot_info[0]["pos"], veck_main, N_p, boxsize
        ),
    ]
    print(f"ZA χ^2={np.mean((psi_div_1 - target_psi_div) ** 2)}")
    print(
        f"ALPT(rs=6) χ^2={np.mean((dl.div_ALPT(LPT2_disp, veck_main, 6, init_delta) - target_psi_div) ** 2)}"
    )

    overdensities = []

    for i in range(len(final_pos)):
        delta = np.zeros((grid_size, grid_size, grid_size), dtype=dl.real_dtype)
        current_pos = final_pos[i]
        # construct 3D density field
        MASL.MA(current_pos, delta, boxsize, "CIC", verbose=False)

        delta = delta / np.mean(delta) - 1
        overdensities.append(delta)


    Pks = [  # compute matter power spectrum
        PKL.Pk(
            overdensities[n],
            boxsize,
            axis=0,
            MAS=MAS,
            threads=threads,
            verbose=False,
        )
        for n in range(len(overdensities))
    ]
    fig = plt.figure(figsize=(10, 5))
    ax = fig.add_subplot(111)
    ax.set_xscale("log")
    ax.set_xlabel(r"$k~[h{\rm Mpc}^{-1}]$")
    ax.set_ylabel(r"$P/P_{N-body}$")
    ax.set_xlim(0.01, 1.0)
    ax.set_ylim(0.0, 1.5)
    ax.set_title("Matter Pow Spectrum at z=0")
    ax.axvline(
        x=veck_main.Nyquist_freq, color="grey", linestyle="--", linewidth=1, alpha=0.7
    )

    print("plotting recovered")

    for n in range(len(overdensities)):
        ax.plot(
            Pks[0].k3D,
            [(Pks[n].Pk[i, 0]) / (Pks[0].Pk[i, 0]) for i in range(len(Pks[0].k3D))],
            label=labels[n],
        )
    plt.legend()

    plt.show() 

drawfigs()